In [1]:
import torch
from diffusers import DDPMScheduler
from models.genre_diffusion_model import GenreConditionedUNet


/home/svshenoy/.local/lib/python3.11/site-packages/diffusers/utils/outputs.py:63: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(
/home/svshenoy/.local/lib/python3.11/site-packages/diffusers/utils/outputs.py:63: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = GenreConditionedUNet().to(device)
model.train()

scheduler = DDPMScheduler(num_train_timesteps=1000)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)



In [3]:
def get_dummy_batch(batch_size, num_genres):
    
    latents = torch.randn(batch_size, 8, 256, 16)
    genre_ids = torch.randint(0, num_genres, (batch_size,))
    
    return latents, genre_ids


In [4]:
#gpt generated dummy loop with dummy data
num_genres = model.genre_condition.num_genres
null_genre = model.genre_condition.null_genre_id

for epoch in range(5):
    latents, genre_ids = get_dummy_batch(5, num_genres)

    latents = latents.to(device).float()
    genre_ids = genre_ids.to(device).long()
    mask = torch.rand(5, device=device) < 0.1
    genre_ids[mask] = null_genre
    timesteps = torch.randint(0,scheduler.config.num_train_timesteps, (5,),device=device ).long()
    noise = torch.randn_like(latents)
    noisy_latents = scheduler.add_noise(
        latents,
        noise,
        timesteps
    )
    pred_noise = model(
        noisy_latents,
        timesteps,
        genre_ids
    )
    loss = torch.nn.functional.mse_loss(pred_noise, noise)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    print(f"Epoch {epoch} | Loss: {loss.item():.4f}")


Epoch 0 | Loss: 1.1149
Epoch 1 | Loss: 1.0405
Epoch 2 | Loss: 1.0146
Epoch 3 | Loss: 0.9094
Epoch 4 | Loss: 0.8834
